# Ciência de Dados Geoespaciais - Modelagem

**Maj Diego - 2° Semestre / 2026**

**Objetivos**

1. Explicar e aplicar modelos de **regressão linear**;
2. Explicar e aplicar modelos de **regressão logística**;
3. Explicar e aplicar modelos de **classificação supervisionada**;
4. Explicar e aplicar modelos de **classificação não-supervisionada (segmentação)**.

## O Contexto

Na etapa de **Modelagem** — vamos *aprender uma função* a partir dos dados para **explicar** relações e **prever** valores em casos novos.

Prospecção $\rightarrow$ Pré-Processamento $\rightarrow$ Exploração $\rightarrow$ **Modelagem** $\rightarrow$ Comunicação

Três famílias de tarefas organizam praticamente todo o aprendizado de máquina clássico:

| Tarefa | Pergunta que responde | Saída | Exemplo geoespacial |
|---|---|---|---|
| **Regressão** | *Quanto?* | número contínuo | Estimar o salário médio de um município |
| **Classificação** | *Qual categoria?* | rótulo discreto | Rotular um município como saneamento Alto/Médio/Baixo |
| **Segmentação** (clusterização) | *Quais grupos existem?* | grupos descobertos | Descobrir agrupamentos socioeconômicos sem rótulos prévios |

A diferença fundamental está na **supervisão**:

- **Aprendizado supervisionado** — os dados de treino já vêm com a "resposta certa" (o alvo $y$). Regressão e classificação são supervisionadas.
- **Aprendizado não-supervisionado** — não há rótulo; o modelo organiza os dados sozinho. É o caso da **segmentação**.

### Dataset de referência

Reutilizamos o mesmo conjunto da aula de Mineração: indicadores da [plataforma IBGE Cidades](https://cidades.ibge.gov.br/brasil/sintese/rj?indicadores=96385,96386,329756,78192,143558,60030) para os **municípios do Rio de Janeiro**, combinados com a malha municipal oficial do IBGE (2022).

A célula abaixo é idêntica à da aula anterior — se o arquivo `cidades_rj.gpkg` já existe na pasta, ele é carregado direto; caso contrário, é reconstruído a partir das APIs do IBGE.

In [1]:
import os
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import seaborn as sns
import requests
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import (mean_absolute_error, r2_score,
                              classification_report, confusion_matrix)

np.random.seed(42)
sns.set_theme(style='whitegrid')

url_cidades = 'https://cidades.ibge.gov.br/brasil/sintese/rj?indicadores=96385,96386,329756,78192,143558,60030'
api_base = 'https://servicodados.ibge.gov.br/api/v1'
indicadores = {
    96385: 'populacao',
    96386: 'dens_demografica_ibge',
    329756: 'idhm',
    78192: 'ideb_anos_finais_publica',
    143558: 'salario_medio_mensal_sm',
    60030: 'esgotamento_sanitario_pct',
}

def valor_mais_recente(serie):
    """Retorna o valor numérico mais recente de uma série anual da API do IBGE."""
    for ano, valor in sorted(serie.items(), key=lambda item: item[0], reverse=True):
        if valor not in (None, '-', '...'):
            return pd.to_numeric(valor, errors='coerce'), int(ano)
    return np.nan, np.nan

def baixar_indicador(indicador_id, nome_coluna):
    endpoint = f'{api_base}/pesquisas/indicadores/{indicador_id}/resultados/N6[N3[33]]'
    resposta = requests.get(endpoint, timeout=30)
    resposta.raise_for_status()
    registros = []
    for item in resposta.json()[0]['res']:
        valor, ano = valor_mais_recente(item['res'])
        registros.append({
            'cod_mun6': str(item['localidade']).zfill(6),
            nome_coluna: valor,
            f'{nome_coluna}_ano': ano,
        })
    return pd.DataFrame(registros)

def baixar_populacao_historica():
    endpoint = f'{api_base}/pesquisas/indicadores/96385/resultados/N6[N3[33]]'
    resposta = requests.get(endpoint, timeout=30)
    resposta.raise_for_status()
    registros = []
    for item in resposta.json()[0]['res']:
        serie = item['res']
        registros.append({
            'cod_mun6': str(item['localidade']).zfill(6),
            'pop_2010': pd.to_numeric(serie.get('2010'), errors='coerce'),
            'pop_2022': pd.to_numeric(serie.get('2022'), errors='coerce'),
        })
    return pd.DataFrame(registros)

def limpar_colunas_ano(gdf):
    """Incorpora colunas *_ano ao nome da variável correspondente e remove o *_ano."""
    renomear = {}
    remover = []
    aliases_analise = {}

    for coluna_ano in sorted(col for col in gdf.columns if col.endswith('_ano')):
        coluna_base = coluna_ano[:-4]
        if coluna_base not in gdf.columns:
            continue

        anos = sorted(gdf[coluna_ano].dropna().astype(int).unique())
        if len(anos) == 1:
            sufixo = str(anos[0])
        elif len(anos) > 1:
            sufixo = str(anos[-1])
            print(f'Aviso: {coluna_ano} possui múltiplos anos {anos}; usando sufixo {sufixo}.')
        else:
            sufixo = 'sem_ano'

        nova_coluna = f'{coluna_base}_{sufixo}'
        renomear[coluna_base] = nova_coluna
        remover.append(coluna_ano)
        aliases_analise[nova_coluna] = coluna_base

    return gdf.rename(columns=renomear).drop(columns=remover), aliases_analise

filename = "cidades_rj.gpkg"
if not os.path.exists(filename):

    dados_ibge = None
    for indicador_id, nome_coluna in indicadores.items():
        parcial = baixar_indicador(indicador_id, nome_coluna)
        dados_ibge = parcial if dados_ibge is None else dados_ibge.merge(parcial, on='cod_mun6', how='outer')
    del dados_ibge['populacao']; del dados_ibge['populacao_ano']

    malha_path = 'BR_Municipios_2022.zip'
    if not os.path.exists(malha_path):
        malha_url = (
            'https://geoftp.ibge.gov.br/organizacao_do_territorio/malhas_territoriais/'
            'malhas_municipais/municipio_2022/Brasil/BR/BR_Municipios_2022.zip'
        )

        with requests.get(malha_url, stream=True) as resposta:
            resposta.raise_for_status()
            with open(malha_path, "wb") as f:
                for bloco in resposta.iter_content(chunk_size=8192):
                    f.write(bloco)

    malha = gpd.read_file(malha_path)
    malha_rj = malha.query("SIGLA_UF == 'RJ'").copy()
    malha_rj['cod_mun6'] = malha_rj['CD_MUN'].astype(str).str[:6]
    del malha_rj['CD_MUN']; del malha_rj['SIGLA_UF']

    gdf = malha_rj.merge(dados_ibge, on='cod_mun6', how='left')
    gdf = gdf.merge(baixar_populacao_historica(), on='cod_mun6', how='left')
    gdf = gdf.rename(columns={'NM_MUN': 'municipio', 'AREA_KM2': 'area_km2'})
    gdf = gdf.to_crs('EPSG:4326')

    # Variáveis derivadas a partir dos dados oficiais.
    gdf['cresc_pop_pct'] = ((gdf['pop_2022'] / gdf['pop_2010'] - 1) * 100).round(2)
    gdf['classe_dev'] = pd.cut(
        gdf['idhm'],
        bins=[0, 0.699, 0.799, 1.0],
        labels=['Médio', 'Alto', 'Muito Alto']
    )
    gdf['classe_ideb'] = pd.cut(
        gdf['ideb_anos_finais_publica'],
            bins=[0, 3.0, 4.0, 5.0, 6.0, 10.0],
            labels=['Muito Baixo', 'Baixo', 'Médio', 'Alto', 'Muito Alto']
        )

    # Limpar anos e renomear colunas dos anos
    gdf, aliases_analise = limpar_colunas_ano(gdf)

    gdf.to_file(filename, driver="GPKG")
else:
    gdf = gpd.read_file(filename)


# print(f'Fonte IBGE Cidades: {url_cidades}')
print(f'Shape gdf limpo: {gdf.shape}')
gdf.head(3)

Aviso: ideb_anos_finais_publica_ano possui múltiplos anos [np.int64(2021), np.int64(2023)]; usando sufixo 2023.
Shape gdf limpo: (92, 14)


,municipio,area_km2,geometry,cod_mun6,dens_demografica_ibge_2022,idhm_2010,ideb_anos_finais_publica_2023,salario_medio_mensal_sm_2024,esgotamento_sanitario_pct_2022,pop_2010,pop_2022,cresc_pop_pct,classe_dev,classe_ideb
0,Angra dos Reis,813.420,"MULTIPOLYGON (((-44.51649 -23.03589, -44.51633...",330010,205.84,0.724,4.4,3.2,69.95,169511,167434,-1.23,Alto,Médio
1,Aperibé,94.542,"POLYGON ((-42.11437 -21.61204, -42.11394 -21.6...",330015,116.71,0.692,5.3,1.7,84.82,10213,11034,8.04,Médio,Alto
2,Araruama,638.276,"POLYGON ((-42.28399 -22.93928, -42.28444 -22.9...",330020,203.16,0.718,4.3,1.7,32.34,112008,129671,15.77,Alto,Médio


Relembrando as variáveis numéricas disponíveis no `gdf`. Elas serão nossos **atributos** (features, o vetor $X$) e, dependendo da tarefa, um deles será o **alvo** ($y$).

In [2]:
num_cols = gdf.describe().columns.tolist()
print('Variáveis numéricas disponíveis:')
for c in num_cols:
    print(' -', c)
gdf[num_cols].describe().T.round(2)

Variáveis numéricas disponíveis:
 - area_km2
 - dens_demografica_ibge_2022
 - idhm_2010
 - ideb_anos_finais_publica_2023
 - salario_medio_mensal_sm_2024
 - esgotamento_sanitario_pct_2022
 - pop_2010
 - pop_2022
 - cresc_pop_pct


,count,mean,std,min,25%,50%,75%,max
area_km2,92.0,475.55,488.22,19.39,215.62,373.61,586.25,4032.49
dens_demografica_ibge_2022,92.0,682.04,1797.64,12.62,51.89,109.48,311.11,12521.64
idhm_2010,92.0,0.71,0.04,0.61,0.68,0.71,0.73,0.84
ideb_anos_finais_publica_2023,92.0,4.50,0.55,3.50,4.07,4.40,4.90,5.80
salario_medio_mensal_sm_2024,92.0,2.08,0.64,1.20,1.80,1.90,2.20,5.60
esgotamento_sanitario_pct_2022,92.0,67.49,20.33,2.86,56.04,70.59,82.74,98.71
pop_2010,92.0,173803.58,672081.03,5269.00,17502.25,34878.50,113948.25,6320446.00
pop_2022,92.0,174512.76,658952.50,5415.00,17476.75,37767.00,132384.00,6211223.00
cresc_pop_pct,92.0,5.09,11.25,-12.87,-0.77,1.84,7.76,54.77


## 1. Regressão Linear

### 1.1 A ideia

A **regressão linear** modela o alvo $y$ como uma **combinação linear** dos atributos:

$$\hat{y} = \beta_0 + \beta_1 x_1 + \beta_2 x_2 + \cdots + \beta_p x_p$$

- $\beta_0$ é o **intercepto** (valor de $\hat y$ quando todos os $x=0$);
- cada $\beta_j$ é o **coeficiente** do atributo $x_j$: *quanto* $\hat y$ varia quando $x_j$ aumenta em uma unidade, mantendo os demais constantes.

O treino consiste em encontrar os $\beta$ que **minimizam a soma dos quadrados dos resíduos** (mínimos quadrados ordinários — OLS):

$$\min_{\beta}\ \sum_{i=1}^{n}\big(y_i - \hat y_i\big)^2$$

**Suposições** que tornam a interpretação válida: relação aproximadamente linear, resíduos independentes, com variância constante (homocedasticidade) e aproximadamente normais.

### 1.2 Caso simples: uma variável

Vamos começar prevendo o **salário médio mensal** (`salario_medio_mensal_sm_2023`, em salários mínimos) a partir do **IDHM** (`idhm_2010`). Uma única variável nos permite visualizar a reta ajustada.

In [3]:
from sklearn.linear_model import LinearRegression

dados = gdf[['municipio', 'idhm_2010', 'salario_medio_mensal_sm_2023']].dropna()
X = dados[['idhm_2010']].values          # atributo (matriz n x 1)
y = dados['salario_medio_mensal_sm_2023'].values  # alvo

modelo_simples = LinearRegression().fit(X, y)

print(f'Intercepto (β0): {modelo_simples.intercept_:.3f}')
print(f'Coeficiente (β1) do IDHM: {modelo_simples.coef_[0]:.3f}')
print(f'R²: {modelo_simples.score(X, y):.3f}')
print(f'\nLeitura: a cada +0,1 no IDHM, o salário médio previsto sobe '
      f'{modelo_simples.coef_[0]*0.1:.2f} salário(s) mínimo(s).')

KeyError: "['salario_medio_mensal_sm_2023'] not in index"

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(dados['idhm_2010'], y, alpha=0.7, edgecolors='white', s=55,
           color='steelblue', label='Municípios')

xs = np.linspace(dados['idhm_2010'].min(), dados['idhm_2010'].max(), 100).reshape(-1, 1)
ax.plot(xs, modelo_simples.predict(xs), color='red', linewidth=2, label='Reta ajustada')

ax.set_xlabel('IDHM (2010)')
ax.set_ylabel('Salário médio mensal (SM)')
ax.set_title('Regressão linear simples: IDHM → Salário médio')
ax.legend()
plt.tight_layout()
plt.show()

### 1.3 Regressão múltipla e avaliação honesta

Na prática combinamos vários atributos. E, para saber se o modelo **generaliza** (não apenas decora o treino), separamos os dados em **treino** e **teste** com `train_test_split`. Métricas de regressão:

- **MAE** (erro absoluto médio): erro típico na unidade do alvo — fácil de interpretar.
- **RMSE** (raiz do erro quadrático médio): penaliza mais os erros grandes.
- **R²** (coeficiente de determinação): fração da variância do alvo explicada pelo modelo (1 = perfeito, 0 = equivale a chutar a média).

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score

features_reg = ['idhm_2010', 'ideb_anos_finais_publica_2023',
                'esgotamento_sanitario_pct_2022', 'dens_demografica_ibge_2022']
alvo_reg = 'salario_medio_mensal_sm_2023'

reg = gdf[features_reg + [alvo_reg]].dropna()
X = reg[features_reg]
y = reg[alvo_reg]

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=42)

modelo = LinearRegression().fit(X_tr, y_tr)
y_pred = modelo.predict(X_te)

mae  = mean_absolute_error(y_te, y_pred)
rmse = np.sqrt(np.mean((y_te - y_pred) ** 2))
r2   = r2_score(y_te, y_pred)

print(f'MAE  = {mae:.3f} SM')
print(f'RMSE = {rmse:.3f} SM')
print(f'R²   = {r2:.3f}')

coefs = pd.Series(modelo.coef_, index=features_reg).sort_values(key=abs, ascending=False)
print('\nCoeficientes (impacto de cada variável):')
print(coefs.round(4))

> **Cuidado com a escala!** Os coeficientes brutos não são diretamente comparáveis porque cada variável está em uma unidade diferente (IDHM vai de 0 a 1; densidade chega a milhares). Para comparar *importâncias*, padronize os atributos (`StandardScaler`) antes de ajustar — aí cada coeficiente mede o efeito em desvios-padrão.

In [ ]:
# Gráfico "previsto vs. observado": quanto mais perto da diagonal, melhor.
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(y_te, y_pred, alpha=0.7, edgecolors='white', s=60, color='steelblue')
lim = [min(y_te.min(), y_pred.min()), max(y_te.max(), y_pred.max())]
ax.plot(lim, lim, 'r--', linewidth=1.5, label='previsão perfeita')
ax.set_xlabel('Salário observado (SM)')
ax.set_ylabel('Salário previsto (SM)')
ax.set_title(f'Previsto vs. Observado  (R² = {r2:.2f})')
ax.legend()
plt.tight_layout()
plt.show()

### 1.4 Extrapolação temporal (previsão de tendência)

Um uso clássico da regressão linear em dados geoespaciais/temporais é **ajustar uma tendência** e projetá-la no futuro. Como exemplo didático, ajustamos a evolução média da **população** do estado entre 2010 e 2022 e projetamos anos seguintes.

> **Atenção metodológica:** com apenas dois pontos (2010 e 2022) a reta passa exatamente por eles — a projeção é uma *ilustração do mecanismo*, não uma previsão demográfica confiável. Com uma série anual completa (vários anos), o mesmo código produz uma estimativa robusta.

In [ ]:
anos_disp = [2010, 2022]
pop_media = [gdf['pop_2010'].mean(), gdf['pop_2022'].mean()]

X_ano = np.array(anos_disp).reshape(-1, 1)
y_pop = np.array(pop_media)

tend = LinearRegression().fit(X_ano, y_pop)
print(f'Taxa média de variação: {tend.coef_[0]:,.0f} habitantes/ano por município')

anos_futuros = np.array([2010, 2022, 2030, 2040]).reshape(-1, 1)
proj = tend.predict(anos_futuros)

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(anos_disp, pop_media, color='steelblue', s=80, zorder=3, label='Observado')
ax.plot(anos_futuros, proj, 'r--', marker='o', label='Tendência / projeção')
for a, p in zip(anos_futuros.flatten(), proj):
    ax.annotate(f'{p:,.0f}', (a, p), textcoords='offset points', xytext=(0, 8), fontsize=8)
ax.set_xlabel('Ano'); ax.set_ylabel('População média por município')
ax.set_title('Ajuste de tendência com regressão linear')
ax.legend()
plt.tight_layout()
plt.show()

## 2. Regressão Logística

### 2.1 De regressão para classificação

Apesar do nome, a **regressão logística é um modelo de classificação**. Em vez de prever um número contínuo, ela estima a **probabilidade de pertencer a uma classe**.

O truque é passar a combinação linear por uma função **sigmoide** (logística), que "espreme" qualquer número real para o intervalo $(0, 1)$:

$$P(y=1 \mid x) = \sigma(z) = \frac{1}{1 + e^{-z}}, \qquad z = \beta_0 + \beta_1 x_1 + \cdots + \beta_p x_p$$

Classificamos como classe 1 quando $P \ge 0{,}5$ (limiar ajustável). O treino maximiza a **verossimilhança** (equivalente a minimizar a *log-loss*).

In [ ]:
# Visualizando a função sigmoide
z = np.linspace(-8, 8, 200)
sig = 1 / (1 + np.exp(-z))
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(z, sig, color='crimson', linewidth=2)
ax.axhline(0.5, color='gray', linestyle=':', linewidth=1)
ax.axvline(0, color='gray', linestyle=':', linewidth=1)
ax.set_xlabel('z  (combinação linear dos atributos)')
ax.set_ylabel('P(classe = 1)')
ax.set_title('Função sigmoide (logística)')
plt.tight_layout()
plt.show()

### 2.2 Aplicação: município com bom saneamento?

Vamos criar um alvo **binário**: o município tem esgotamento sanitário **acima da mediana** do estado (1) ou não (0)? Depois treinamos a logística para prever isso a partir de indicadores socioeconômicos.

Note dois cuidados de boa prática:
- **Padronização** dos atributos (`StandardScaler`) dentro de um `Pipeline`, para o modelo convergir bem;
- **Estratificação** no split (`stratify=y`), para manter a proporção das classes em treino e teste.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, roc_auc_score, confusion_matrix, ConfusionMatrixDisplay

# Alvo binário derivado do saneamento
mediana_esg = gdf['esgotamento_sanitario_pct_2022'].median()
gdf['bom_saneamento'] = (gdf['esgotamento_sanitario_pct_2022'] > mediana_esg).astype(int)

features_log = ['idhm_2010', 'ideb_anos_finais_publica_2023',
                'salario_medio_mensal_sm_2023', 'dens_demografica_ibge_2022']
dados_log = gdf[features_log + ['bom_saneamento']].dropna()
X = dados_log[features_log]
y = dados_log['bom_saneamento']

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25,
                                          random_state=42, stratify=y)

clf_log = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))
clf_log.fit(X_tr, y_tr)

y_pred = clf_log.predict(X_te)
y_prob = clf_log.predict_proba(X_te)[:, 1]

print(f'Acurácia : {accuracy_score(y_te, y_pred):.3f}')
print(f'AUC-ROC  : {roc_auc_score(y_te, y_prob):.3f}')

In [ ]:
# Matriz de confusão: acertos na diagonal, erros fora dela
cm = confusion_matrix(y_te, y_pred)
fig, ax = plt.subplots(figsize=(5, 4))
ConfusionMatrixDisplay(cm, display_labels=['Não', 'Sim']).plot(ax=ax, cmap='Blues', colorbar=False)
ax.set_title('Matriz de confusão — "Bom saneamento?"')
ax.set_xlabel('Previsto'); ax.set_ylabel('Real')
plt.tight_layout()
plt.show()

### 2.3 Interpretando os coeficientes (razão de chances)

Na logística, exponenciar um coeficiente dá a **razão de chances** (*odds ratio*): quanto as chances da classe positiva se multiplicam quando o atributo (padronizado) aumenta em 1 desvio-padrão. Valor > 1 empurra para a classe 1; < 1 empurra para a classe 0.

In [ ]:
coef = clf_log.named_steps['logisticregression'].coef_[0]
razao_chances = np.exp(coef)
tabela = pd.DataFrame({
    'coeficiente': coef,
    'razao_de_chances': razao_chances
}, index=features_log).sort_values('razao_de_chances', ascending=False)
print(tabela.round(3))

## 3. Classificação Supervisionada

A logística já é um classificador, mas existe uma família rica de métodos que aprendem **fronteiras de decisão** mais flexíveis. O fluxo é sempre o mesmo:

1. definir atributos $X$ e alvo categórico $y$;
2. dividir em treino/teste;
3. treinar (`.fit`) e prever (`.predict`);
4. avaliar com métricas apropriadas.

| Método | Ideia central | Ponto forte |
|---|---|---|
| **Árvore de Decisão** | Regras sucessivas do tipo "se $x_j > t$..." | Interpretável, sem precisar padronizar |
| **KNN** | Classe majoritária dos $k$ vizinhos mais próximos | Simples; exige padronização |
| **Random Forest** | Média de muitas árvores diversas | Robusto, boa acurácia, mede importância |
| **SVM** | Fronteira de margem máxima entre classes | Bom em alta dimensão |

### 3.1 Definindo um alvo multiclasse

Vamos criar três **faixas de saneamento** por tercis (`pd.qcut`) — o que garante classes balanceadas — e treinar modelos para prever a faixa a partir de outros indicadores.

In [ ]:
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score
from sklearn.metrics import classification_report

gdf['faixa_saneamento'] = pd.qcut(gdf['esgotamento_sanitario_pct_2022'],
                                  q=3, labels=['Baixa', 'Média', 'Alta'])

features_cls = ['idhm_2010', 'salario_medio_mensal_sm_2023',
                'ideb_anos_finais_publica_2023', 'dens_demografica_ibge_2022']
dados_cls = gdf[features_cls + ['faixa_saneamento']].dropna()
X = dados_cls[features_cls]
y = dados_cls['faixa_saneamento'].astype(str)

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.30,
                                          random_state=42, stratify=y)
print('Distribuição das classes:')
print(y.value_counts())

### 3.2 Comparando modelos

Usamos **validação cruzada** (`cross_val_score`, 5 folds) além da acurácia de teste: ela dá uma estimativa mais estável ao repartir os dados de várias formas.

In [ ]:
modelos = {
    'Árvore de Decisão': DecisionTreeClassifier(max_depth=3, random_state=42),
    'KNN (k=5)': make_pipeline(StandardScaler(), KNeighborsClassifier(n_neighbors=5)),
    'Random Forest': RandomForestClassifier(n_estimators=200, random_state=42),
}

resultados = []
for nome, modelo in modelos.items():
    modelo.fit(X_tr, y_tr)
    acc_teste = accuracy_score(y_te, modelo.predict(X_te))
    acc_cv = cross_val_score(modelo, X, y, cv=5).mean()
    resultados.append({'modelo': nome, 'acc_teste': acc_teste, 'acc_cv': acc_cv})

pd.DataFrame(resultados).set_index('modelo').round(3)

### 3.3 Interpretabilidade: a árvore e a importância de variáveis

A grande vantagem da árvore é que podemos **ler** suas regras. Já a Random Forest, embora seja uma "caixa mais fechada", fornece a **importância de cada atributo** — quanto cada variável contribuiu para separar as classes.

In [ ]:
arvore = DecisionTreeClassifier(max_depth=3, random_state=42).fit(X_tr, y_tr)

fig, ax = plt.subplots(figsize=(14, 6))
plot_tree(arvore, feature_names=features_cls,
          class_names=arvore.classes_, filled=True, rounded=True, fontsize=8, ax=ax)
ax.set_title('Árvore de decisão — faixa de saneamento')
plt.tight_layout()
plt.show()

In [ ]:
rf = RandomForestClassifier(n_estimators=200, random_state=42).fit(X_tr, y_tr)

imp = pd.Series(rf.feature_importances_, index=features_cls).sort_values()
fig, ax = plt.subplots(figsize=(8, 4))
imp.plot.barh(ax=ax, color='seagreen')
ax.set_title('Importância das variáveis (Random Forest)')
ax.set_xlabel('Importância relativa')
plt.tight_layout()
plt.show()

print('Relatório de classificação (Random Forest, conjunto de teste):')
print(classification_report(y_te, rf.predict(X_te)))

### 3.4 Onde os modelos erram, no mapa

Como estamos em ciência de dados **geoespaciais**, vale plotar os acertos e erros no espaço — às vezes os erros se concentram em uma região, revelando um padrão que os atributos não capturaram.

In [ ]:
gdf_cls = gdf.loc[dados_cls.index].copy()
gdf_cls['pred_rf'] = rf.predict(dados_cls[features_cls])
gdf_cls['acertou'] = gdf_cls['pred_rf'] == gdf_cls['faixa_saneamento'].astype(str)

fig, axes = plt.subplots(1, 2, figsize=(13, 6))
gdf_cls.plot(column='faixa_saneamento', ax=axes[0], legend=True,
             cmap='YlGnBu', edgecolor='black', linewidth=0.3)
axes[0].set_title('Faixa de saneamento (real)'); axes[0].set_axis_off()

gdf_cls.plot(column='acertou', ax=axes[1], legend=True,
             cmap='RdYlGn', edgecolor='black', linewidth=0.3,
             categorical=True)
axes[1].set_title('Acerto do Random Forest'); axes[1].set_axis_off()
plt.tight_layout()
plt.show()

## 4. Classificação Não-Supervisionada (Segmentação)

Aqui **não há rótulo**. O objetivo é **descobrir grupos** (*clusters*/segmentos) de municípios semelhantes entre si e diferentes dos demais. Já vimos K-médias e DBSCAN na aula de Mineração; agora fechamos o ciclo mostrando como **escolher o número de grupos**, **avaliar** a qualidade sem rótulos e **caracterizar** os segmentos encontrados.

| Método | Como agrupa | Precisa dizer nº de grupos? |
|---|---|---|
| **K-médias** | minimiza a distância aos centros dos grupos | sim (o *k*) |
| **DBSCAN** | agrupa por densidade; marca isolados como ruído | não (usa `eps`, `min_samples`) |
| **Agrupamento hierárquico** | funde grupos progressivamente (dendrograma) | não (corta a árvore) |

> **Passo obrigatório:** como esses métodos usam distâncias, **padronize** os atributos antes (`StandardScaler`); caso contrário, variáveis de grande magnitude (densidade) dominam as de pequena (IDHM).

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score

features_seg = ['idhm_2010', 'salario_medio_mensal_sm_2023',
                'ideb_anos_finais_publica_2023', 'esgotamento_sanitario_pct_2022']
dados_seg = gdf[features_seg].dropna()

scaler = StandardScaler()
X_scaled = scaler.fit_transform(dados_seg)

### 4.1 Quantos grupos? Cotovelo + Silhueta

Dois critérios complementares:
- **Método do cotovelo** — a *inércia* (soma das distâncias internas) cai à medida que aumentamos $k$; procuramos o "joelho" onde o ganho passa a ser pequeno.
- **Coeficiente de silhueta** — mede o quão bem cada ponto se encaixa em seu grupo (−1 a 1; **quanto maior, melhor**). Escolhemos o $k$ de maior silhueta.

In [ ]:
Ks = range(2, 8)
inercias, silhuetas = [], []
for k in Ks:
    km = KMeans(n_clusters=k, random_state=42, n_init=10).fit(X_scaled)
    inercias.append(km.inertia_)
    silhuetas.append(silhouette_score(X_scaled, km.labels_))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(list(Ks), inercias, 'o-', color='steelblue')
axes[0].set_title('Método do Cotovelo'); axes[0].set_xlabel('k'); axes[0].set_ylabel('Inércia')

axes[1].plot(list(Ks), silhuetas, 'o-', color='crimson')
axes[1].set_title('Coeficiente de Silhueta'); axes[1].set_xlabel('k'); axes[1].set_ylabel('Silhueta')
melhor_k = list(Ks)[int(np.argmax(silhuetas))]
axes[1].axvline(melhor_k, color='gray', linestyle=':')
plt.tight_layout()
plt.show()

print(f'Melhor k pela silhueta: {melhor_k}')

### 4.2 Ajustando e caracterizando os segmentos

Com o $k$ escolhido, ajustamos o K-médias e — passo essencial — **interpretamos** os grupos: calculamos o perfil médio de cada segmento para dar-lhes um *significado* (ex.: "municípios de alto desenvolvimento", "periferia de baixa cobertura").

In [ ]:
km = KMeans(n_clusters=melhor_k, random_state=42, n_init=10).fit(X_scaled)
dados_seg = dados_seg.copy()
dados_seg['segmento'] = km.labels_

# Perfil médio de cada segmento (nas unidades originais)
perfil = dados_seg.groupby('segmento')[features_seg].mean().round(2)
perfil['n_municipios'] = dados_seg['segmento'].value_counts().sort_index()
print('Perfil médio de cada segmento:')
perfil

In [ ]:
# Projeção em 2D via PCA só para visualizar os grupos
from sklearn.decomposition import PCA
pcs = PCA(n_components=2).fit_transform(X_scaled)

fig, ax = plt.subplots(figsize=(8, 6))
sc = ax.scatter(pcs[:, 0], pcs[:, 1], c=km.labels_, cmap='viridis',
                s=60, alpha=0.8, edgecolors='white')
ax.set_xlabel('PC1'); ax.set_ylabel('PC2')
ax.set_title(f'Segmentos de municípios (K-médias, k={melhor_k})')
plt.colorbar(sc, label='segmento')
plt.tight_layout()
plt.show()

### 4.3 O mapa dos segmentos

O produto final da segmentação geoespacial é o **mapa**: onde estão, no território, os grupos descobertos? Padrões contíguos sugerem processos regionais (a autocorrelação espacial que medimos com o Índice de Moran na aula anterior).

In [ ]:
gdf_seg = gdf.loc[dados_seg.index].copy()
gdf_seg['segmento'] = dados_seg['segmento'].values

fig, ax = plt.subplots(figsize=(9, 7))
gdf_seg.plot(column='segmento', ax=ax, legend=True, categorical=True,
             cmap='viridis', edgecolor='black', linewidth=0.3)
ax.set_title(f'Segmentação socioeconômica dos municípios do RJ (k={melhor_k})')
ax.set_axis_off()
plt.tight_layout()
plt.show()

## Síntese: qual modelo usar?

```
                      Tenho a "resposta certa" (alvo y) nos dados de treino?
                                        │
                 ┌──────────────────────┴───────────────────────┐
                SIM (supervisionado)                     NÃO (não-supervisionado)
                 │                                              │
      O alvo é número ou categoria?                      Quero descobrir grupos
                 │                                       → SEGMENTAÇÃO
        ┌────────┴─────────┐                             (K-médias, DBSCAN,
     NÚMERO             CATEGORIA                          hierárquico)
   → REGRESSÃO        → CLASSIFICAÇÃO
   (linear, RF,       (logística, árvore,
    SVR, ...)          KNN, RF, SVM, ...)
```

**Boas práticas transversais que aplicamos:**
- separar **treino/teste** (e usar **validação cruzada**) para medir generalização, não memorização;
- **padronizar** atributos para modelos baseados em distância (KNN, K-médias) ou gradiente (logística);
- escolher a **métrica certa** para a tarefa (MAE/R² na regressão; acurácia/AUC/relatório na classificação; silhueta na segmentação);
- em dados geoespaciais, **levar os resultados ao mapa** — erros e grupos costumam ter estrutura espacial.

## Lista de exercícios

Continue usando os dados municipais do seu estado (IBGE Cidades + malha municipal), como na aula de Mineração.

**Exercício 1 — Regressão linear.** Escolha um alvo numérico (ex.: `salario_medio_mensal_sm`) e ajuste uma regressão múltipla com pelo menos 3 atributos. Reporte MAE, RMSE e R² no conjunto de teste e interprete o coeficiente de maior magnitude.

**Exercício 2 — Regressão logística.** Crie um alvo binário a partir de um indicador (ex.: IDEB acima/abaixo da mediana) e treine uma logística. Apresente a matriz de confusão, a AUC e a interpretação das razões de chances.

**Exercício 3 — Classificação supervisionada.** Defina um alvo com 3 classes (`pd.qcut`) e compare Árvore, KNN e Random Forest por validação cruzada. Plote a árvore e a importância das variáveis, e leve os acertos/erros ao mapa.

**Exercício 4 — Segmentação.** Rode K-médias variando $k$ de 2 a 7, escolha o $k$ pela silhueta, caracterize o perfil de cada segmento e produza o mapa. Compare visualmente os grupos com o Índice de Moran calculado na aula anterior.

**Exercício 5 (Desafio) — Regressão para projeção.** Com a série histórica decenal de cobertura de esgoto do seu estado (do [ipeadata](https://www.ipeadata.gov.br/Default.aspx)), ajuste uma regressão linear sobre a **média estadual** e estime em que ano o estado atingirá **95%** de cobertura. Discuta as limitações da extrapolação linear.